## Introduction

This notebook develops user interface menus options can be used to retrieve relevant WDL tasks for use in RAG.

We need user input in order to select the right WDL tasks to build a worfkflow. We need to know what kind of input data they have (e.g. paired-end DNA FASTQs), their analysis goals (e.g. variant calling), and if they prefer any specific tools (e.g. bwa-mem, strelka). Our ChromaDB RAG database 

We want to avoid free-text input for many reasons, inclding to prevent users from generating toxic output or leaking protected data. Therefore, we will give users preset options that can be mapped to metadata keywords in our database. In a point-and-click interface these would be dropdown menus. For this MVP we will use a command-line interface with numbered options.

## 1. Imports

In [1]:
import chromadb

## 2. Decide metadata to filter on

Think about the metadata categories we want to map to user input options and use for keyword filtering. An example of metadata for a single "document"/WDL task in our database :

```{python}
    metadatas=[{
        "tool": "strelka",
        "task": "strelka_germline",
        "topic": ["genomics", "dna_polymorphism"],
        "species": ["eukaryote"],
        "operation": "variant_calling",
        "input_sample_data_types": ["nucleic_acid_sequence_alignment", "data_index"],
        "input_sample_format_types": ["bam", "bai"],
        "output_sample_data_types": ["sequence_variations", "data_index"]
    }]
```

Note that some or all of this metadata can be fed to the LLM as additional context along with the WDL task.

**I think the relevant categories are:**
- `tool`: User may have bioinformatics preferences
- `topic`: Whether the data is DNA/RNA/protein, etc.
- `species`: Some tools are restricted to certain species
- `operation`: What does the user want the WDL to do
- `input_sample_format_types`: The format usually tells us the data type, especially as input data

Also `task` may be relevant for tools like GATK where the tasks are tools unto themselves.

## 3. Map metadata terms to user-friendly ones

Most of the metadata keyword terms come from the EDAM ontology so that they are consistent and descriptive. However, many of these terms are not ones researchers would use in daily conversation (e.g. many would say "aligned data" instead of "nucleic acid sequence alignment").

We need to decide on user-facing terms taht map to metadata keywords for these categories:
- `topic`
- `operation`
- `input_sample_format_types`

The `tool` and `species` categories are self-explanatory and don't use EDAM terms (note that for tools like GATK we may want to present the individual `task`s as "tool" options).

First, we need to look at the terms we have for each category.

### `topic`

In [ ]:
def get_collection(chroma_dir, collection_name="wdl_tasks"):
    client = chromadb.PersistentClient(path=chroma_dir)
    return client.get_collection(name=collection_name)

collection = get_collection('../data/chroma/')

In [ ]:
import chromadb

client = chromadb.Client()  # or however you initialize your client
collection = client.get_collection("your_collection_name")

results = collection.get(include=["metadatas"])

unique_topics = set()
for metadata in results["metadatas"]:
    topics = metadata.get("topic", [])
    if isinstance(topics, list):
        unique_topics.update(topics)
    elif isinstance(topics, str):
        unique_topics.add(topics)

print(sorted(unique_topics))


In [ ]:
fsdf